# Подготовка данных и синтетического причинного слоя

Этот ноутбук открывает экспериментальный контур работы. Его задача - превратить исходные финансовые таблицы Home Credit в воспроизводимый набор данных для проверки uplift-подхода в финтехе.

Логика ноутбука:

1. собрать и очистить кредитные признаки;
2. сохранить связь с классической задачей риска через `TARGET`;
3. добавить синтетический слой клиентских воздействий;
4. явно разделить безопасные признаки, treatment, outcome, потенциальные исходы и oracle-поля.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler

Объявим вспомогательную функцию для преобразования многоуровневых названий колонок в плоский формат.


In [2]:
def flatten_columns(df):
    df.columns = [
        "_".join(col).upper() if isinstance(col, tuple) else col
        for col in df.columns
    ]
    return df

## 1. Загрузка исходных данных

На этом шаге загружаются исходные таблицы Home Credit. В общей логике работы они дают реалистичную финансовую основу: демографию, параметры заявки, кредитную историю и признаки просрочки.

Коммуникаций в исходном наборе нет. Поэтому ниже поверх этих признаков будет построен синтетический причинный слой, который нужен не для имитации конкретного банка, а для управляемой проверки uplift-методики.

In [3]:
import os

DATA_PATH = "data/raw"       # сырые CSV файлы
PROCESSED_PATH = "data/processed"  # обработанные данные
FEATURES_PATH = "features"   # описания признаков

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(FEATURES_PATH, exist_ok=True)

Зафиксируем случайность (random seed) для жесткой воспроизводимости пайплайна.


In [4]:
np.random.seed(91)

Считаем основную таблицу заявок.


In [5]:
app = pd.read_csv(DATA_PATH + "/application_train.csv")

print("Application shape:", app.shape)

Application shape: (307511, 122)


## 2. Агрегация вспомогательных таблиц

Вспомогательные таблицы агрегируются до уровня клиента. Это сохраняет банковскую природу данных и позволяет затем сравнивать риск-ориентированные модели с моделями индивидуального эффекта на одном признаковом пространстве.

In [6]:
bureau = pd.read_csv(DATA_PATH + "/bureau.csv")

bureau_agg = bureau.groupby("SK_ID_CURR").agg({
    "DAYS_CREDIT": ["mean", "min", "max"],
    "AMT_CREDIT_SUM": ["mean", "sum"],
    "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],
    "AMT_CREDIT_SUM_OVERDUE": ["sum"],
    "CREDIT_DAY_OVERDUE": ["max"],
})

bureau_agg = flatten_columns(bureau_agg).reset_index()

print("bureau_agg:", bureau_agg.shape)

bureau_agg: (305811, 10)


In [7]:
prev = pd.read_csv(DATA_PATH + "/previous_application.csv")

prev_agg = prev.groupby("SK_ID_CURR").agg({
    "AMT_APPLICATION": ["mean", "max"],
    "AMT_CREDIT": ["mean", "max"],
    "CNT_PAYMENT": ["mean"],
    "DAYS_DECISION": ["mean"],
})

prev_agg = flatten_columns(prev_agg).reset_index()

print("prev_agg:", prev_agg.shape)

prev_agg: (338857, 7)


In [8]:
pos = pd.read_csv(DATA_PATH + "/POS_CASH_balance.csv")

pos_agg = pos.groupby("SK_ID_CURR").agg({
    "MONTHS_BALANCE": ["mean", "min"],
    "CNT_INSTALMENT": ["mean"],
    "CNT_INSTALMENT_FUTURE": ["mean"],
    "SK_DPD": ["max"],
})

pos_agg = flatten_columns(pos_agg).reset_index()

print("pos_agg:", pos_agg.shape)

pos_agg: (337252, 6)


In [9]:
inst = pd.read_csv(DATA_PATH + "/installments_payments.csv")

inst_agg = inst.groupby("SK_ID_CURR").agg({
    "AMT_PAYMENT": ["mean", "sum"],
    "AMT_INSTALMENT": ["mean"],
    "DAYS_ENTRY_PAYMENT": ["mean"],
    "DAYS_INSTALMENT": ["mean"],
})

inst_agg = flatten_columns(inst_agg).reset_index()

print("inst_agg:", inst_agg.shape)

inst_agg: (339587, 6)


In [10]:
credit = pd.read_csv(DATA_PATH + "/credit_card_balance.csv")

credit_agg = credit.groupby("SK_ID_CURR").agg({
    "AMT_BALANCE": ["mean", "max"],
    "AMT_DRAWINGS_CURRENT": ["mean"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean"],
    "SK_DPD": ["max"],
})

credit_agg = flatten_columns(credit_agg).reset_index()

print("credit_agg:", credit_agg.shape)

credit_agg: (103558, 6)


In [11]:
df = app.copy()

df = df.merge(bureau_agg, on="SK_ID_CURR", how="left")
df = df.merge(prev_agg, on="SK_ID_CURR", how="left")
df = df.merge(pos_agg, on="SK_ID_CURR", how="left")
df = df.merge(inst_agg, on="SK_ID_CURR", how="left")
df = df.merge(credit_agg, on="SK_ID_CURR", how="left")

print("Итоговый набор данных:", df.shape)

Итоговый набор данных: (307511, 152)


## 3. Предобработка набора данных

Предобработка нужна для двух целей: подготовить данные для классического скоринга и создать стабильную основу для синтетического причинного слоя. Здесь важно не смешивать исходные признаки клиента с будущими uplift-переменными.

In [12]:
df = df.replace([np.inf, -np.inf], np.nan)

# Создаем копию набора данных, в df_sim обработаем пропуски для корректной генерации переменных индивидуального эффекта воздействия, при этом сохраним пропуски в исходном df
df_sim = df.copy()

df_sim = df_sim.fillna(df_sim.median(numeric_only=True))

## 4. Анализ признаков объединенного набора данных

Этот блок проверяет, что после агрегации и предобработки набор остается пригодным для задачи кредитного риска. Сначала анализируется исходная целевая переменная `TARGET`; затем поверх нее строится отдельная задача воздействия.

In [13]:
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,AMT_PAYMENT_MEAN,AMT_PAYMENT_SUM,AMT_INSTALMENT_MEAN,DAYS_ENTRY_PAYMENT_MEAN,DAYS_INSTALMENT_MEAN,AMT_BALANCE_MEAN,AMT_BALANCE_MAX,AMT_DRAWINGS_CURRENT_MEAN,AMT_CREDIT_LIMIT_ACTUAL_MEAN,SK_DPD_MAX_y
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,11559.247105,219625.695,11559.247105,-315.421053,-295.000000,NaN,NaN,NaN,NaN,NaN
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,64754.586000,1618864.650,64754.586000,-1385.320000,-1378.160000,NaN,NaN,NaN,NaN,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,7096.155000,21288.465,7096.155000,-761.666667,-754.000000,NaN,NaN,NaN,NaN,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,62947.088438,1007153.415,62947.088438,-271.625000,-252.250000,0.0,0.0,0.0,270000.0,0.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,12214.060227,806127.975,12666.444545,-1032.242424,-1028.606061,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,7492.924286,52450.470,7492.924286,-156.285714,-120.000000,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,10069.867500,60419.205,10069.867500,-2393.833333,-2391.000000,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,4115.915357,57622.815,4399.707857,-2387.428571,-2372.928571,NaN,NaN,NaN,NaN,NaN
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,10239.832895,194556.825,10239.832895,-161.263158,-142.263158,NaN,NaN,NaN,NaN,NaN


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 152 entries, SK_ID_CURR to SK_DPD_MAX_y
dtypes: float64(95), int64(41), object(16)
memory usage: 356.6+ MB


### Описание целевой переменной TARGET

`TARGET` используется как исходный индикатор кредитного риска в Home Credit. Он нужен для калибровки и проверки реалистичности `BASE_PD`, но не является итоговой целевой переменной uplift-эксперимента.

В дальнейшем итоговый outcome для коммуникаций задается как `TARGET_AFTER_CONTACT`: он отражает дефолт после фактически назначенного канала.

In [15]:
df['TARGET'].value_counts(normalize=True).round(4) * 100

TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64

In [16]:
dt_cols = df.select_dtypes(include=['datetime64']).columns.to_list()
object_cols = df.select_dtypes(include=['object']).columns.to_list()
numeric_cols = df.select_dtypes(include=['number']).columns.to_list()
flg_cols = [col for col in numeric_cols if "FLAG_" in col]

numeric_cols = sorted(list(set(numeric_cols) - set(flg_cols)))

numeric_cols.remove('TARGET')

In [17]:
dt_cols

[]

In [18]:
object_cols

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [19]:
numeric_cols

['AMT_ANNUITY',
 'AMT_APPLICATION_MAX',
 'AMT_APPLICATION_MEAN',
 'AMT_BALANCE_MAX',
 'AMT_BALANCE_MEAN',
 'AMT_CREDIT',
 'AMT_CREDIT_LIMIT_ACTUAL_MEAN',
 'AMT_CREDIT_MAX',
 'AMT_CREDIT_MEAN',
 'AMT_CREDIT_SUM_DEBT_MEAN',
 'AMT_CREDIT_SUM_DEBT_SUM',
 'AMT_CREDIT_SUM_MEAN',
 'AMT_CREDIT_SUM_OVERDUE_SUM',
 'AMT_CREDIT_SUM_SUM',
 'AMT_DRAWINGS_CURRENT_MEAN',
 'AMT_GOODS_PRICE',
 'AMT_INCOME_TOTAL',
 'AMT_INSTALMENT_MEAN',
 'AMT_PAYMENT_MEAN',
 'AMT_PAYMENT_SUM',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'APARTMENTS_AVG',
 'APARTMENTS_MEDI',
 'APARTMENTS_MODE',
 'BASEMENTAREA_AVG',
 'BASEMENTAREA_MEDI',
 'BASEMENTAREA_MODE',
 'CNT_CHILDREN',
 'CNT_FAM_MEMBERS',
 'CNT_INSTALMENT_FUTURE_MEAN',
 'CNT_INSTALMENT_MEAN',
 'CNT_PAYMENT_MEAN',
 'COMMONAREA_AVG',
 'COMMONAREA_MEDI',
 'COMMONAREA_MODE',
 'CREDIT_DAY_OVERDUE_MAX',
 'DAYS_BIRTH',
 'DAYS_CREDIT_M

In [20]:
flg_cols

['FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'FLAG_DOCUMENT_2',
 'FLAG_DOCUMENT_3',
 'FLAG_DOCUMENT_4',
 'FLAG_DOCUMENT_5',
 'FLAG_DOCUMENT_6',
 'FLAG_DOCUMENT_7',
 'FLAG_DOCUMENT_8',
 'FLAG_DOCUMENT_9',
 'FLAG_DOCUMENT_10',
 'FLAG_DOCUMENT_11',
 'FLAG_DOCUMENT_12',
 'FLAG_DOCUMENT_13',
 'FLAG_DOCUMENT_14',
 'FLAG_DOCUMENT_15',
 'FLAG_DOCUMENT_16',
 'FLAG_DOCUMENT_17',
 'FLAG_DOCUMENT_18',
 'FLAG_DOCUMENT_19',
 'FLAG_DOCUMENT_20',
 'FLAG_DOCUMENT_21']

## 5. Устранение выявленных аномалий

Аномалии корректируются до генерации причинного слоя. Это снижает риск того, что синтетический эффект будет построен на технических выбросах, а не на содержательных финансовых различиях между клиентами.

In [21]:
df = df.rename(columns={
    "SK_DPD_MAX_x": "SK_DPD_MAX_POS",
    "SK_DPD_MAX_y": "SK_DPD_MAX_CC"
})


id_cols = ["SK_ID_CURR"]

In [22]:
flg_cols = [
    col for col in df.columns
    if col.startswith("FLAG_")
] + [
    "LIVE_CITY_NOT_WORK_CITY",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION"
]


target_cols = ["TARGET"]

uplift_cols = [
    "BASE_PD",
    "CONTACT_PROPENSITY",
    "COMMUNICATION",
    "RISK_SEGMENT",
    "CONTACT_HISTORY",
    "PREFERRED_CHANNEL",
    "INTERACTION_SCORE",
    "DELAY_FLAG",
    "PD_NO_CONTACT",
    "PD_SMS",
    "PD_ROBOT_CALL",
    "PD_OPERATOR_CALL",
    "UPLIFT_SMS",
    "UPLIFT_ROBOT_CALL",
    "UPLIFT_OPERATOR_CALL",
    "TRUE_UPLIFT",
    "PD_AFTER_CONTACT",
    "TARGET_AFTER_CONTACT",
    "ORACLE_COMMUNICATION",
    "ORACLE_TRUE_UPLIFT",
    "ORACLE_PD_AFTER_CONTACT",
]


In [23]:
exclude_cols = set(
    id_cols +
    uplift_cols +
    flg_cols +
    object_cols +
    target_cols
)

numeric_cols = [
    col for col in df.columns
    if col not in exclude_cols
]

In [24]:
all_grouped = set(
    id_cols +
    uplift_cols +
    flg_cols +
    object_cols +
    numeric_cols +
    target_cols
)

missing_cols = set(df.columns) - all_grouped
extra_cols = all_grouped - set(df.columns)

print("Пропущенные:", missing_cols)
print("Лишние:", extra_cols)

Пропущенные: set()
Лишние: {'ORACLE_COMMUNICATION', 'PD_OPERATOR_CALL', 'CONTACT_HISTORY', 'UPLIFT_ROBOT_CALL', 'PD_SMS', 'PD_AFTER_CONTACT', 'UPLIFT_SMS', 'UPLIFT_OPERATOR_CALL', 'TRUE_UPLIFT', 'RISK_SEGMENT', 'PREFERRED_CHANNEL', 'ORACLE_TRUE_UPLIFT', 'INTERACTION_SCORE', 'TARGET_AFTER_CONTACT', 'PD_ROBOT_CALL', 'ORACLE_PD_AFTER_CONTACT', 'COMMUNICATION', 'PD_NO_CONTACT', 'CONTACT_PROPENSITY', 'BASE_PD', 'DELAY_FLAG'}


In [25]:
print(f"Всего {len(id_cols)} id cols, {len(uplift_cols)} uplift_cols, {len(flg_cols)} flg_cols, {len(object_cols)} object_cols, {len(numeric_cols)} numeric_cols, {len(target_cols)} target_cols")

Всего 1 id cols, 21 uplift_cols, 34 flg_cols, 16 object_cols, 102 numeric_cols, 1 target_cols


## 6. Синтетическая генерация переменных индивидуального эффекта воздействия

Главный методический шаг ноутбука - добавление слоя коммуникаций. Он превращает задачу из обычного прогнозирования дефолта в задачу выбора воздействия: кому не писать, кому отправить SMS, кому сделать автоматический звонок, а кому нужен оператор.

Слой специально содержит барьеры, характерные для финтеха: неслучайное назначение контакта, слабый эффект, неоднородность реакции, усталость от контактов и разные каналы.

### 6.1 Типы коммуникационных воздействий

В эксперименте используются четыре варианта политики: отсутствие контакта, SMS, автоматический звонок и звонок оператора. Это минимальная многоканальная постановка, где уже возникает выбор между стоимостью канала и ожидаемым снижением риска.

| Значение | Описание |
|---|---|
| `control` | Отсутствие коммуникации (контрольная группа) |
| `sms` | SMS-сообщение с напоминанием о платеже |
| `robot_call` | Автоматический голосовой звонок |
| `operator_call` | Звонок живого оператора контакт-центра |

На практике назначение типа воздействия выполняется **не случайным образом**, а на основе скора клиента, который оценивает его платежеспособность и формируется с помощью моделей машинного обучения, что приводит к характерному для банковских данных смещению: в группу клиентов, которым назначают коммуникации, чаще попадают более рисковые клиенты, что затрудняет прямую оценку эффекта


### 6.2 Логика генерации причинного слоя

Генерация устроена как последовательность управляемых шагов.

1. `BASE_PD` задает риск дефолта без коммуникации.
2. `CONTACT_PROPENSITY` задает вероятность того, что банк вообще инициирует контакт.
3. `COMMUNICATION` назначает канал неслучайно, создавая selection bias.
4. `CONTACT_HISTORY`, `PREFERRED_CHANNEL`, `INTERACTION_SCORE` и `DELAY_FLAG` создают неоднородность реакции.
5. `UPLIFT_*` и `PD_*` сохраняют потенциальные эффекты и исходы для каждого канала.
6. `TARGET_AFTER_CONTACT` фиксирует наблюдаемый outcome после фактически назначенной коммуникации.

Такой слой нужен, чтобы в дальнейшем проверять не только качество моделей, но и условия применимости uplift-подхода.

### 6.3 Реализация функций генерации

Ниже функции идут в том же порядке, что и причинная логика: сначала базовый риск, затем склонность банка к контакту, затем назначение канала, индивидуальные модификаторы эффекта и итоговые потенциальные исходы.

#### `generate_base_pd(df)` — базовая вероятность дефолта

Вычисляет латентную вероятность дефолта клиента без учёта коммуникаций. Используется линейная комбинация наиболее информативных признаков: `EXT_SOURCE_1/2/3`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_INCOME_TOTAL`, `SK_DPD_MAX`, `CREDIT_DAY_OVERDUE_MAX`. Результат нормализуется через сигмоид-функцию. Калибровка mean(BASE_PD) = 0.08 выполняется итеративным бинарным поиском смещения: прямое использование `logit(0.08)` как смещения не корректно из-за неравенства Йенсена — `mean(sigmoid(x)) > sigmoid(mean(x))` для широких распределений.


In [26]:
def generate_base_pd(df):
    """
    Вычисляет базовую вероятность дефолта как линейную комбинацию
    ключевых признаков, нормализованную через сигмоид.

    Калибровка: итеративный бинарный поиск смещения score,
    чтобы mean(BASE_PD) == target_rate (8%) с точностью 1e-5.
    Простой сдвиг на logit(target_rate) не работает из-за
    неравенства Йенсена: mean(sigmoid(x)) > sigmoid(mean(x)).
    """
    features = [
        "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
        "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
        "SK_DPD_MAX_x", "CREDIT_DAY_OVERDUE_MAX",
    ]
    X = df[features].copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    weights = np.array([-0.7, -0.7, -0.7,  0.2,  0.2, -0.2,  0.5,  0.5])
    score = X_scaled @ weights
    score += np.random.normal(0, 0.5, len(df))

    # Итеративная калибровка: ищем сдвиг, при котором mean(BASE_PD) = 0.08
    target_rate = 0.08
    centered = score - score.mean()
    lo, hi = -8.0, 0.0
    for _ in range(60):
        mid = (lo + hi) / 2
        mean_pd = (1 / (1 + np.exp(-(centered + mid)))).mean()
        if mean_pd < target_rate:
            lo = mid
        else:
            hi = mid
    score = centered + (lo + hi) / 2

    df["BASE_PD"] = 1 / (1 + np.exp(-score))
    return df


#### `generate_contact_propensity(df)` — вероятность того, что назначенная клиенту коммуникация будет совершена успешно (клиент примет звонок, получит смс и т.д)

Моделирует политику банка по инициации коммуникаций:

```
CONTACT_PROPENSITY = 0.6 * BASE_PD + 0.2 * (AMT_CREDIT / max) + 0.2 * (AMT_ANNUITY / max)
```

Клиенты с высоким риском и крупными кредитами получают более высокую оценку склонности к воздействию (вероятность назначения коммуникации). Это намеренно создаёт **систематическое смещение отбора**: группа воздействия систематически отличается от контрольной по уровню риска.


In [27]:
def generate_contact_propensity(df):
  '''Policy банка (propensity к контакту)'''
  p = (
        0.6 * df["BASE_PD"] +
        0.2 * (df["AMT_CREDIT"] / df["AMT_CREDIT"].max()) +
        0.2 * (df["AMT_ANNUITY"] / df["AMT_ANNUITY"].max())
    )

  df["CONTACT_PROPENSITY"] = np.clip(p, 0, 1)

  return df

#### `generate_treatment(df)` — назначение типа воздействия

Назначает тип коммуникации на основе оценки склонности(`CONTACT_PROPENSITY`) через вероятностную модель:

| CONTACT_PROPENSITY | Вероятности [none, sms, robot, operator] |
|---|---|
| > 0.7 (высокий риск) | [0.10, 0.20, 0.30, 0.40] |
| 0.4 – 0.7 (средний риск) | [0.25, 0.35, 0.25, 0.15] |
| < 0.4 (низкий риск) | [0.55, 0.30, 0.10, 0.05] |

Механизм имитирует реальную стратегию контакт-центра: дорогие по себестоимости каналы коммуникации назначаются высокорисковым клиентам.


In [28]:
def generate_segments(df):
  '''Сегментация клиентов'''

  df["RISK_SEGMENT"] = pd.qcut(
      df["BASE_PD"],
      q=3,
      labels=["low_risk", "medium_risk", "high_risk"]
  )

  return df

#### `generate_segments(df)` и `generate_contact_history(df)` — сегментация и история контактов

**Сегментация**: клиенты делятся на три группы по квантилям BASE_PD: `low_risk`, `medium_risk`, `high_risk`. Сегмент используется для моделирования гетерогенного эффект воздействия.

**История контактов**: количество предыдущих контактов моделируется из распределения Пуассона с параметром `λ = 0.5 + 3 × BASE_PD`. Более рисковые клиенты контактируются чаще. Значения ограничены диапазоном [0, 10].


In [29]:
def generate_contact_history(df):
  '''Сегментация клиентов'''
  lam = (
      0.5 +
      3 * df["BASE_PD"]
  )

  df["CONTACT_HISTORY"] = np.random.poisson(lam)

  df["CONTACT_HISTORY"] = np.clip(df["CONTACT_HISTORY"], 0, 10)

  return df

#### `fatigue_multiplier(history)` — эффект насыщения контактами

Эффективность коммуникации снижается экспоненциально с ростом истории контактов:

```
fatigue = exp(-0.35 * CONTACT_HISTORY)
```

При большом количестве предыдущих контактов мультипликатор стремится к нулю, что соответствует феномену **эффект насыщения**.


In [30]:
def generate_channel_preference(df):
  '''Channel preference - У клиента есть “любимый” канал коммуникации'''

  channels = ["sms", "robot_call", "operator_call"]

  df["PREFERRED_CHANNEL"] = np.random.choice(
      channels,
      size=len(df),
      p=[0.5, 0.3, 0.2]
  )

  return df

#### `generate_channel_preference(df)` и `generate_interactions(df)` — предпочтения и взаимодействия

**Channel preference**: каждому клиенту назначается предпочтительный для него канал коммуникации (SMS — 50%, robot_call — 30%, operator_call — 20%). Эффект коммуникации усиливается при совпадении назначенного и предпочтительного канала.

**Interaction score**: `INTERACTION_SCORE = income_norm × (1 − credit_norm)`. Клиенты с высоким доходом и низкой кредитной нагрузкой имеют более высокий score, что модифицирует величину эффект воздействия (гетерогенность эффекта).


In [31]:
def generate_interactions(df):
  '''воздействие interaction (гетерогенность эффекта) - Некоторые признаки усиливают эффект коммуникации'''

  income_norm = df["AMT_INCOME_TOTAL"] / df["AMT_INCOME_TOTAL"].max()
  credit_norm = df["AMT_CREDIT"] / df["AMT_CREDIT"].max()

  df["INTERACTION_SCORE"] = (
      0.5 * income_norm +
      0.5 * (1 - credit_norm)
  )

  return df

#### `generate_delay(df)` — отложенная реакция клиента

Часть клиентов реагирует на коммуникацию с задержкой. Вероятность задержки: `delay_prob = 0.3 + 0.4 × BASE_PD`. При `DELAY_FLAG = 1` непосредственный эффект воздействия снижается вдвое.


In [32]:
def generate_delay(df):
  '''Delayed response - Некоторые клиенты реагируют только через время'''

  delay_prob = (
      0.3 +
      0.4 * df["BASE_PD"]
  )

  df["DELAY_FLAG"] = np.random.binomial(1, delay_prob)

  return df

#### Потенциальные исходы и итоговая сборка

Финальный слой хранит два типа информации. Наблюдаемая часть используется как обычные исторические данные: фактический канал и фактический исход. Контрфактическая часть используется только для оценки качества и оракульных сравнений.

Это разграничение принципиально: если oracle-поля попадут в признаки модели, задача станет утечкой, а не честной проверкой uplift.

## 8. Итоговый синтетический набор данных

Итоговый набор совмещает реальные финансовые признаки и синтетический причинный слой. Он позволяет сравнивать три уровня решений:

1. классический прогноз риска;
2. ранжирование клиентов по ожидаемому эффекту фактически назначенной политики;
3. идеализированный выбор лучшего канала через oracle-поля.

В реальном банке третий уровень недоступен, но в исследовательском стенде он нужен как верхняя граница и как способ проверить, восстанавливают ли модели правильный порядок клиентов.

In [33]:
TREATMENT_CHANNELS = ["sms", "robot_call", "operator_call"]
CHANNEL_SUFFIX = {
    "sms": "SMS",
    "robot_call": "ROBOT_CALL",
    "operator_call": "OPERATOR_CALL",
}


def calculate_channel_uplift(df, channel):
    """
    Вычисляет потенциальный индивидуальный эффект воздействия для одного канала коммуникации.

    Значение — изменение вероятности дефолта относительно отсутствия контакта.
    Отрицательный индивидуальный эффект воздействия полезен (снижает PD). Положительный — вреден:
    он соответствует сценарию «sleeping dog» (коммуникация ухудшает поведение клиента).
    """
    base_pd = df["BASE_PD"].to_numpy()
    effect = np.zeros(len(df), dtype=float)

    # Ставки эффекта по сегментам риска: эффект пропорционален BASE_PD,
    # поэтому у низкорисковых клиентов абсолютный эффект меньше.
    segment_rates = {
        "sms": {
            "low_risk": -0.45,
            "medium_risk": -0.12,
            "high_risk": 0.10,
        },
        "robot_call": {
            "low_risk": -0.08,
            "medium_risk": -0.55,
            "high_risk": -0.08,
        },
        "operator_call": {
            "low_risk": 0.04,
            "medium_risk": -0.18,
            "high_risk": -0.22,
        },
    }

    for segment, rate in segment_rates[channel].items():
        segment_mask = df["RISK_SEGMENT"].eq(segment).to_numpy()
        effect[segment_mask] = rate * base_pd[segment_mask]

    # Насыщение контактами ослабляет эффект по экспоненте.
    fatigue = np.exp(-0.35 * df["CONTACT_HISTORY"].to_numpy())
    effect *= fatigue

    # Совпадение назначенного канала с предпочтительным усиливает эффект.
    preferred_channel = df["PREFERRED_CHANNEL"].eq(channel).to_numpy()
    effect[preferred_channel] *= 1.5

    # Interaction score масштабирует эффект по характеристикам клиента.
    effect *= (0.7 + df["INTERACTION_SCORE"].to_numpy())

    # При отложенной реакции немедленный эффект снижается вдвое.
    delayed_response = df["DELAY_FLAG"].eq(1).to_numpy()
    effect[delayed_response] *= 0.5

    # При очень большой истории контактов эффект может стать слегка положительным (sleeping dog).
    saturated = df["CONTACT_HISTORY"].gt(5).to_numpy()
    effect[saturated] += np.random.normal(0.02, 0.01, saturated.sum())

    # Добавляем случайный шум — SNR намеренно слабый, как в реальных CRM-данных.
    effect += np.random.normal(0, 0.005, len(df))
    return effect


def generate_potential_outcomes(df):
    """
    Генерирует контрфактические вероятности дефолта и индивидуальный эффект воздействия для всех каналов.

    Эти поля — оракульная информация, доступная исследователю только благодаря
    синтетическому эксперименту. В признаки модели они включаться не должны.
    """
    df["PD_NO_CONTACT"] = df["BASE_PD"]

    pd_by_treatment = {"control": df["PD_NO_CONTACT"].to_numpy()}
    for channel in TREATMENT_CHANNELS:
        suffix = CHANNEL_SUFFIX[channel]
        uplift_col = f"UPLIFT_{suffix}"
        pd_col = f"PD_{suffix}"

        df[uplift_col] = calculate_channel_uplift(df, channel)
        df[pd_col] = np.clip(df["PD_NO_CONTACT"] + df[uplift_col], 0, 1)
        pd_by_treatment[channel] = df[pd_col].to_numpy()

    pd_matrix = pd.DataFrame(pd_by_treatment, index=df.index)
    df["ORACLE_COMMUNICATION"] = pd_matrix.idxmin(axis=1)
    df["ORACLE_PD_AFTER_CONTACT"] = pd_matrix.min(axis=1)
    df["ORACLE_TRUE_UPLIFT"] = df["ORACLE_PD_AFTER_CONTACT"] - df["PD_NO_CONTACT"]
    return df


def generate_observed_outcome(df):
    """
    Выбирает наблюдаемый эффект и исход в соответствии с назначенным воздействие.

    `TRUE_UPLIFT` совместим с предыдущей версией набора данных: это эффект
    фактически назначенной коммуникации, а не наилучший оракульный эффект.
    """
    pd_cols = {
        "control": "PD_NO_CONTACT",
        "sms": "PD_SMS",
        "robot_call": "PD_ROBOT_CALL",
        "operator_call": "PD_OPERATOR_CALL",
    }
    uplift_cols = {
        "control": None,
        "sms": "UPLIFT_SMS",
        "robot_call": "UPLIFT_ROBOT_CALL",
        "operator_call": "UPLIFT_OPERATOR_CALL",
    }

    df["TRUE_UPLIFT"] = 0.0
    df["PD_AFTER_CONTACT"] = df["PD_NO_CONTACT"].to_numpy()

    for воздействие, pd_col in pd_cols.items():
        mask = df["COMMUNICATION"].eq(воздействие)
        df.loc[mask, "PD_AFTER_CONTACT"] = df.loc[mask, pd_col]

        uplift_col = uplift_cols[воздействие]
        if uplift_col is not None:
            df.loc[mask, "TRUE_UPLIFT"] = df.loc[mask, uplift_col]

    df["TARGET_AFTER_CONTACT"] = np.random.binomial(1, df["PD_AFTER_CONTACT"])
    return df


In [34]:
def generate_uplift_dataset(df):
    """
    Последовательно применяет все генераторы синтетических индивидуальный эффект воздействия-переменных.

    Порядок вызовов важен: каждая последующая функция использует переменные,
    созданные предыдущей (например, assign_communication требует BASE_PD,
    CONTACT_PROPENSITY и RISK_SEGMENT).
    """
    df = generate_base_pd(df)
    df = generate_contact_propensity(df)
    df = generate_segments(df)
    df = generate_contact_history(df)
    df = generate_channel_preference(df)
    df = generate_interactions(df)
    df = generate_delay(df)
    return df


In [35]:
def assign_communication(df):
    """
    Назначает тип коммуникации (воздействие) на основе CONTACT_PROPENSITY
    и RISK_SEGMENT, имитируя реальную стратегию банка.
    """
    # 1. Определяем, будет ли вообще контакт с клиентом (Bernoulli trial)
    # Используем CONTACT_PROPENSITY как вероятность успеха
    is_contacted = np.random.binomial(1, df["CONTACT_PROPENSITY"])

    # 2. Создаем массив для типов коммуникации, по умолчанию "control" (нет контакта)
    comm_types = np.full(len(df), "control", dtype=object)

    # 3. Для тех, с кем контакт состоялся, выбираем тип согласно бизнес-логике [2]:
    # - Высокий риск -> звонок оператора
    # - Средний риск -> робот
    # - Низкий риск -> SMS

    high_mask = (is_contacted == 1) & (df["RISK_SEGMENT"] == "high_risk")
    mid_mask = (is_contacted == 1) & (df["RISK_SEGMENT"] == "medium_risk")
    low_mask = (is_contacted == 1) & (df["RISK_SEGMENT"] == "low_risk")

    comm_types[high_mask] = "operator_call"
    comm_types[mid_mask] = "robot_call"
    comm_types[low_mask] = "sms"

    df["COMMUNICATION"] = comm_types

    return df

In [36]:
# Итоговая последовательность вызовов:
df_sim = generate_uplift_dataset(df_sim)       # Создаем латентные признаки и сегменты
df_sim = generate_potential_outcomes(df_sim)   # Считаем PD/эффект воздействия для всех возможных каналов
df_sim = assign_communication(df_sim)          # Назначаем фактически наблюдаемое воздействие
df_sim = generate_observed_outcome(df_sim)     # Выбираем наблюдаемый эффект и бинарный исход

C:\Users\sharn\AppData\Local\Temp\ipykernel_70716\349974783.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["BASE_PD"] = 1 / (1 + np.exp(-score))
C:\Users\sharn\AppData\Local\Temp\ipykernel_70716\3249116891.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["CONTACT_PROPENSITY"] = np.clip(p, 0, 1)
C:\Users\sharn\AppData\Local\Temp\ipykernel_70716\2526531180.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

In [37]:
observed_summary = pd.DataFrame({
    "rows": df_sim.groupby("COMMUNICATION").size(),
    "mean_base_pd": df_sim.groupby("COMMUNICATION")["BASE_PD"].mean(),
    "mean_true_uplift": df_sim.groupby("COMMUNICATION")["TRUE_UPLIFT"].mean(),
    "mean_pd_after": df_sim.groupby("COMMUNICATION")["PD_AFTER_CONTACT"].mean(),
    "target_after_rate": df_sim.groupby("COMMUNICATION")["TARGET_AFTER_CONTACT"].mean(),
}).round(4)

print("Сводка по наблюдаемым воздействиям:")
print(observed_summary)

print("Распределение каналов по оракулу (%):")
print((df_sim["ORACLE_COMMUNICATION"].value_counts(normalize=True) * 100).round(2))


Сводка по наблюдаемым воздействиям:
                 rows  mean_base_pd  mean_true_uplift  mean_pd_after  \
COMMUNICATION                                                          
control        277048        0.0683            0.0000         0.0683   
operator_call   17558        0.3043           -0.0347         0.2695   
robot_call       7617        0.0361           -0.0177         0.0185   
sms              5288        0.0098           -0.0044         0.0060   

               target_after_rate  
COMMUNICATION                     
control                   0.0684  
operator_call             0.2664  
robot_call                0.0163  
sms                       0.0043  
Распределение каналов по оракулу (%):
ORACLE_COMMUNICATION
operator_call    38.28
robot_call       38.11
sms              21.43
control           2.18
Name: proportion, dtype: float64


### 8.1 Разделение признаков по роли и защита от утечки

После генерации все переменные разделяются по роли. Безопасные исходные признаки могут использоваться в моделях. Treatment, outcome, потенциальные исходы и oracle-поля используются только там, где это предусмотрено постановкой.

Это одна из ключевых практических идей работы: uplift-модель полезна только при честном дизайне данных. Если модель видит будущий outcome или истинный эффект, качество перестает иметь прикладной смысл.

In [38]:
synthetic_context_cols = [
    "BASE_PD",
    "CONTACT_PROPENSITY",
    "RISK_SEGMENT",
    "CONTACT_HISTORY",
    "PREFERRED_CHANNEL",
    "INTERACTION_SCORE",
    "DELAY_FLAG",
]

treatment_cols = ["COMMUNICATION"]
observed_outcome_cols = ["PD_AFTER_CONTACT", "TARGET_AFTER_CONTACT"]

potential_outcome_cols = [
    "PD_NO_CONTACT",
    "PD_SMS",
    "PD_ROBOT_CALL",
    "PD_OPERATOR_CALL",
    "UPLIFT_SMS",
    "UPLIFT_ROBOT_CALL",
    "UPLIFT_OPERATOR_CALL",
]

oracle_cols = [
    "TRUE_UPLIFT",
    "ORACLE_COMMUNICATION",
    "ORACLE_TRUE_UPLIFT",
    "ORACLE_PD_AFTER_CONTACT",
]

# Эти поля нельзя использовать как признаки в обычных предиктивных моделях.
# Часть синтетических контекстных переменных допустима в эффект воздействия-экспериментах,
# только если эксперимент явно указывает, что они известны до воздействия.
leakage_cols_for_modeling = sorted(set(
    synthetic_context_cols +
    treatment_cols +
    observed_outcome_cols +
    potential_outcome_cols +
    oracle_cols +
    ["TARGET"]
))

synthetic_output_cols = [
    col for col in (
        synthetic_context_cols +
        treatment_cols +
        potential_outcome_cols +
        oracle_cols +
        observed_outcome_cols
    )
    if col in df_sim.columns
]

feature_role_groups = {
    "safe_raw_feature_note": "All original Home Credit columns except IDs, targets and generated причинно-следственный/oracle columns.",
    "synthetic_context_cols": synthetic_context_cols,
    "treatment_cols": treatment_cols,
    "observed_outcome_cols": observed_outcome_cols,
    "potential_outcome_cols": potential_outcome_cols,
    "oracle_cols": oracle_cols,
    "leakage_cols_for_modeling": leakage_cols_for_modeling,
}

with open(os.path.join(FEATURES_PATH, "feature_role_groups.json"), "w", encoding="utf-8") as f:
    json.dump(feature_role_groups, f, ensure_ascii=False, indent=2)

print(f"Синтетические переменные: {len(synthetic_output_cols)}")
print(f"Переменные, которые нельзя использовать в построении моделей: {len(leakage_cols_for_modeling)}")

df_sim = df_sim.copy()

Синтетические переменные: 21
Переменные, которые нельзя использовать в построении моделей: 22


In [39]:
# Переносим синтетические поля в набор данных с оригинальными пропусками.
# Исходные признаки Home Credit остаются в df без медианной импутации;
# импутация использовалась только внутри df_sim для стабильной генерации.
df[synthetic_output_cols] = df_sim[synthetic_output_cols]


In [40]:
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,PD_OPERATOR_CALL,UPLIFT_SMS,UPLIFT_ROBOT_CALL,UPLIFT_OPERATOR_CALL,TRUE_UPLIFT,ORACLE_COMMUNICATION,ORACLE_TRUE_UPLIFT,ORACLE_PD_AFTER_CONTACT,PD_AFTER_CONTACT,TARGET_AFTER_CONTACT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.671466,0.021933,0.016229,-0.005122,-0.005122,operator_call,-0.005122,0.671466,0.671466,1
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.047954,0.008737,-0.022656,0.000660,0.000000,robot_call,-0.022656,0.024638,0.047294,0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.008536,-0.010511,-0.003049,0.002265,0.000000,sms,-0.006270,0.000000,0.006270,0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.014554,-0.005826,0.001288,-0.000989,0.000000,sms,-0.005826,0.009717,0.015543,0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.080671,0.005212,-0.004084,-0.019559,0.000000,operator_call,-0.019559,0.080671,0.100230,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0.069975,0.021921,-0.009142,-0.027362,0.000000,operator_call,-0.027362,0.069975,0.097337,0
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0.077837,0.018272,-0.001632,-0.023324,0.000000,operator_call,-0.023324,0.077837,0.101161,0
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0.016405,-0.014803,0.003480,0.001136,0.000000,sms,-0.014803,0.000467,0.015270,0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0.017641,0.000952,-0.007578,-0.006032,0.000000,robot_call,-0.007578,0.016094,0.023673,0


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 173 entries, SK_ID_CURR to TARGET_AFTER_CONTACT
dtypes: category(1), float64(109), int32(3), int64(41), object(19)
memory usage: 400.3+ MB


In [42]:
# Сохраняем как CSV
df.to_csv(os.path.join(PROCESSED_PATH, "uplift-dataset.csv"), index=False)